# Preprocessing

We use the German dataset (Available at https://github.com/SustainEval/sustaineval2025_data/) from the SustainEval GermEval shared task (Available at https://sustaineval.github.io/) by Prange et al. (2025). It is based on the German Sustainability Code (Deutscher Nachhaltigkeitskodex, Available at https://www.deutscher-nachhaltigkeitskodex.de/de/ueber-den-dnk/nutzen-des-dnk/), where companies can voluntarily publish their ESG reports. Please download the files into the [data](data) folder before running this notebook.

In [21]:
import pandas as pd
from collections import defaultdict, Counter
import random
import numpy as np
import json

### Combine understandability annotations with SustainEval GermEval dataset

In [22]:
dataset_train = pd.read_json("data/training_data.jsonl", lines=True)
annotations_train = pd.read_json("data/annotations_train.jsonl", lines=True)

dataset_dev = pd.read_json("data/development_data.jsonl", lines=True)
annotations_dev = pd.read_json("data/annotations_dev.jsonl", lines=True)

dataset_eval = pd.read_json("data/evaluation_data.jsonl", lines=True)
annotations_eval = pd.read_json("data/annotations_eval.jsonl", lines=True)

In [23]:
def add_annotations(dataset, annotations):
    # Merge the dataset with the annotations on the 'id' column
    merged_df = pd.merge(dataset, annotations, on="id", how="inner")
    return merged_df

dataset_train = add_annotations(dataset_train, annotations_train)
dataset_dev = add_annotations(dataset_dev, annotations_dev)
dataset_eval = add_annotations(dataset_eval, annotations_eval)

In [24]:
# Rename target column to text
dataset_train = dataset_train.rename(columns={"target": "text"})
dataset_dev = dataset_dev.rename(columns={"target": "text"})
dataset_eval = dataset_eval.rename(columns={"target": "text"})

# Filter out unnecessary columns
dataset_train = dataset_train.filter(["id", "text", "understandability"])
dataset_dev = dataset_dev.filter(["id", "text", "understandability"])
dataset_eval = dataset_eval.filter(["id", "text", "understandability"])

### Majority vote as label

In [25]:
# Use majority vote to calculate the label for each sample

def majority_vote(data):

    # Find all elements with maximum frequency
    count = Counter(data)
    max_count = max(list(count.values()))
    modes = [k for k, v in count.items() if v == max_count]

    # If there is more than one mode, return the mean of the modes
    if len(modes) > 1:
        return np.mean(modes)
    else:
        return float(modes[0])


dataset_train["label"] = dataset_train["understandability"].apply(majority_vote)
dataset_dev["label"] = dataset_dev["understandability"].apply(majority_vote)
dataset_eval["label"] = dataset_eval["understandability"].apply(majority_vote)

dataset_train = dataset_train.drop(columns=["understandability"])
dataset_dev = dataset_dev.drop(columns=["understandability"])
dataset_eval = dataset_eval.drop(columns=["understandability"])

In [26]:
# convert to dict
dataset_train = dataset_train.set_index("id").T.to_dict()
dataset_dev = dataset_dev.set_index("id").T.to_dict()
dataset_eval = dataset_eval.set_index("id").T.to_dict()

### Oversample and shuffle

In [27]:
def oversample_data(data: dict[str, dict]) -> dict[str, dict]:
    """
    Oversample the data to have the same number of samples for each class.
    We first use as many samples as possible from the original data and then we randomly draw addictional ones as needed
    """
    
    # Count the occurrences of each label
    temp_labels = [row['label'] for row in data.values()]

    int_labels = [label for label in temp_labels if label == float(round(label))] # remove all non-integer labels
    
    other_labels = [label for label in temp_labels if label != float(round(label))] # remove all integer labels
    
    label_counts = Counter(int_labels)
    print(label_counts)

    
    # Find the majority class and its count
    majority_class = label_counts.most_common(1)[0][0]
    majority_count = label_counts[majority_class]
    
    oversampling_dict = {}

    # Oversample all classes to have at least majority_count samples per class
    for label, count in label_counts.items():
        num_duplicates = majority_count - count
        if num_duplicates > 0:

            # Add the original samples to the oversampling_dict
            for id_, value in data.items():
                if value['label'] == label:
                    oversampling_dict[id_] = value
            
            # Randomly select samples to duplicate
            sample_ids = [id for id, data in data.items() if data['label'] == label]
            oversampled_samples = random.choices(sample_ids, k=num_duplicates)

            for i, sample in enumerate(oversampled_samples):
                new_id = f"{sample}_OVERSAMPLED_{i}"
                oversampling_dict[new_id] = data[sample]

        else:
            # If the class is already in majority, just add it to the oversampling_dict
            for id_, value in data.items():
                if value['label'] == label:
                    oversampling_dict[id_] = value

    # Add the samples from the other labels
    for label in other_labels:
        for id_, value in data.items():
            if value['label'] == label:
                oversampling_dict[id_] = value

    return oversampling_dict



# only oversample the training data
dataset_train_oversampled = oversample_data(dataset_train)

Counter({4.0: 671, 3.0: 167, 2.0: 21, 1.0: 5})


In [28]:
def shuffle_data(data: dict):
    """
    We shuffle the data, as otherwise they would be ordered by the labels after oversampling.
    """
    
    keys = list(data.keys()) # Extract keys
    random.shuffle(keys) # Shuffle keys    
    shuffled_data = {key: data[key] for key in keys} # Rebuild dict with shuffled keys
    
    return shuffled_data

dataset_train = shuffle_data(dataset_train)
dataset_train_oversampled = shuffle_data(dataset_train_oversampled)
dataset_dev = shuffle_data(dataset_dev)
dataset_eval = shuffle_data(dataset_eval)

### Scale the labels

In [29]:
def scale(value: float):
    """
    Expecting a value range from 1 - 4 --> we scale to 0 - 1
    """

    value = float(value)  # Ensure value is a float

    return (value - 1) / 3.0

def scale_labels(data: dict) -> dict:
    """
    Scales the labels according to current scale function.
    """

    labels = [value['label'] for value in data.values() if 'label' in value]

    new_data = {}

    for key, value in data.items():
        new_data[key] = value.copy()
        new_data[key]['label'] = scale(value['label'])

    labels = [value['label'] for value in new_data.values() if 'label' in value]
    
    return new_data

dataset_train = scale_labels(dataset_train)
dataset_train_oversampled = scale_labels(dataset_train_oversampled)
dataset_dev = scale_labels(dataset_dev)
dataset_eval = scale_labels(dataset_eval)

### Save the preprocessed files

In [30]:
def save_jsonl(data, filename):
    with open(filename, 'w') as f:
        for id, item in data.items():
            new_item = {"id": id}
            new_item.update(item)
            json_line = json.dumps(new_item)
            f.write(json_line + '\n')

save_jsonl(dataset_train, "data/preprocessed_annotation_train_original.jsonl") # this is only for the analysis
save_jsonl(dataset_train_oversampled, "data/preprocessed_annotation_train.jsonl") # this will be used for training
save_jsonl(dataset_dev, "data/preprocessed_annotation_dev.jsonl")
save_jsonl(dataset_eval, "data/preprocessed_annotation_eval.jsonl")